# Titanic Survival Prediction: A Machine Learning Tutorial

This notebook walks through a complete machine learning workflow to predict passenger survival on the Titanic. We will use a **Support Vector Classifier (SVC)** and focus on best practices like data cleaning, pipeline building, and hyperparameter tuning with Grid Search.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


from sklearn.svm import SVC

## 1. Data Loading and Initial Exploration

We start by loading our raw data. Exploratory Data Analysis (EDA) is crucial because it helps us identify missing values and understand the data types (numerical vs. categorical) we need to handle later.

In [3]:
# Load the dataset from a CSV file into a pandas DataFrame
df = pd.read_csv("titanic_data_updated.csv")

# Display 5 random rows to see what the data looks like
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
246,247,no,third,"Lindahl, Miss. Agda Thorilda Viktoria",female,25.0,0,0,347071,7.7750,NaN,S
247,248,yes,second,"Hamalainen, Mrs. William (Anna)",female,24.0,0,2,250649,14.5000,NaN,S
361,362,no,second,"del Carlo, Mr. Sebastiano",male,29.0,1,0,SC/PARIS 2167,27.7208,NaN,C
708,709,yes,first,"Cleaver, Miss. Alice",female,22.0,0,0,113781,151.5500,NaN,S
691,692,yes,third,"Karun, Miss. Manca",female,4.0,0,1,349256,13.4167,NaN,C


In [4]:
df['Cabin'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: Cabin
Non-Null Count  Dtype 
--------------  ----- 
204 non-null    object
dtypes: object(1)
memory usage: 7.1+ KB


In [5]:
df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
df['Cabin'] = df['Cabin'].fillna("Missing")

df['Deck'] = df['Cabin'].astype(str).str[0]
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
128,129,yes,third,"Peter, Miss. Anna",female,NaN,1,1,2668,22.3583,F E69,C,3,F
721,722,no,third,"Jensen, Mr. Svend Lauritz",male,17.0,1,0,350048,7.0542,Missing,S,2,M
534,535,no,third,"Cacic, Miss. Marija",female,30.0,0,0,315084,8.6625,Missing,S,1,M
107,108,yes,third,"Moss, Mr. Albert Johan",male,NaN,0,0,312991,7.7750,Missing,S,1,M
762,763,yes,third,"Barah, Mr. Hanna Assi",male,20.0,0,0,2663,7.2292,Missing,C,1,M


In [6]:
df['Deck'].value_counts()

Deck
M    687
C     59
B     47
D     33
E     32
A     15
F     13
G      4
T      1
Name: count, dtype: int64

In [9]:
# Separate the features (X) from the target we want to predict (y)
# We drop 'Survived' from X because that's our answer key
X = df.drop('Survived', axis=1)

# y contains only the survival status
y = df['Survived']